In [17]:
import os
import sqlite3

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver


In [10]:
!uv pip install langchain

Using Python 3.14.6 environment at: E:\AI-ML\.venv
Resolved 36 packages in 2.47s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 6 packages in 500ms
 + langchain==1.3.17
 + langgraph==1.2.11
 + langgraph-checkpoint==4.2.0
 + langgraph-prebuilt==1.1.0
 + langgraph-sdk==0.4.3
 + ormsgpack==1.12.2


In [18]:


model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",   # or "gemini-1.5-pro", "gemini-1.5-flash"
    google_api_key=os.getenv("GOOGLE_API_KEY"),  # or set GOOGLE_API_KEY env var
)


In [14]:
me = {"configurable": {"thread_id": "another_thread_test"}} 

In [19]:
DB_PATH = "study_assistant.db"


def init_db():
    conn = sqlite3.connect(DB_PATH)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS notes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            text TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

    conn.commit()
    conn.close()


init_db()

print("Database ready.")

Database ready.


In [20]:
@tool
def save_note(text: str) -> str:
    """
    Save an important study note into SQLite.
    """

    conn = sqlite3.connect(DB_PATH)

    conn.execute(
        "INSERT INTO notes (text) VALUES (?)",
        (text,)
    )

    conn.commit()
    conn.close()

    return "Note saved successfully."

In [21]:
@tool
def list_notes() -> str:
    """
    Show all study notes saved in SQLite.
    """

    conn = sqlite3.connect(DB_PATH)

    cursor = conn.execute("""
        SELECT id, text, created_at
        FROM notes
        ORDER BY id ASC
    """)

    rows = cursor.fetchall()

    conn.close()

    if not rows:
        return "No notes saved yet."

    result = []

    for note_id, text, created_at in rows:
        result.append(
            f"ID: {note_id} | {text} | {created_at}"
        )

    return "\n".join(result)

In [22]:
@tool
def delete_note(note_id: int) -> str:
    """
    Delete one study note using its ID.
    """

    conn = sqlite3.connect(DB_PATH)

    cursor = conn.execute(
        "DELETE FROM notes WHERE id = ?",
        (note_id,)
    )

    conn.commit()

    deleted = cursor.rowcount

    conn.close()

    if deleted:
        return f"Note {note_id} deleted."

    return f"Note {note_id} does not exist."

In [23]:
def init_tasks_table():

    conn = sqlite3.connect(DB_PATH)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS tasks (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            task TEXT NOT NULL,
            status TEXT DEFAULT 'pending'
        )
    """)

    conn.commit()
    conn.close()


init_tasks_table()

In [24]:
@tool
def add_task(task: str) -> str:
    """
    Add a study task that needs to be completed.
    """

    conn = sqlite3.connect(DB_PATH)

    cursor = conn.execute(
        """
        INSERT INTO tasks (task)
        VALUES (?)
        """,
        (task,)
    )

    task_id = cursor.lastrowid

    conn.commit()
    conn.close()

    return f"Task {task_id} added: {task}"

In [25]:
@tool
def list_tasks() -> str:
    """
    Show all study tasks.
    """

    conn = sqlite3.connect(DB_PATH)

    cursor = conn.execute("""
        SELECT id, task, status
        FROM tasks
        ORDER BY id ASC
    """)

    rows = cursor.fetchall()

    conn.close()

    if not rows:
        return "No tasks."

    result = []

    for task_id, task, status in rows:
        result.append(
            f"{task_id}. {task} [{status}]"
        )

    return "\n".join(result)

In [26]:
@tool
def complete_task(task_id: int) -> str:
    """
    Mark a study task as completed.
    """

    conn = sqlite3.connect(DB_PATH)

    cursor = conn.execute(
        """
        UPDATE tasks
        SET status = 'completed'
        WHERE id = ?
        """,
        (task_id,)
    )

    conn.commit()

    updated = cursor.rowcount

    conn.close()

    if updated:
        return f"Task {task_id} completed."

    return f"Task {task_id} not found."

In [27]:
TOOLS = [
    save_note,
    list_notes,
    delete_note,
    add_task,
    list_tasks,
    complete_task
]

In [28]:
checkpointer = InMemorySaver()

assistant = create_agent(
    model=model,
    tools=TOOLS,
    checkpointer=checkpointer
)

In [29]:
config = {
    "configurable": {
        "thread_id": "aayush-study"
    }
}

In [30]:
def ask_agent(text):

    response = assistant.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": text
                }
            ]
        },
        config
    )

    print("Agent:")
    print(response["messages"][-1].content)

In [31]:
ask_agent(
    "Remember that I am currently learning LangGraph."
)

Agent:
[{'type': 'text', 'text': "I've made a note that you are currently learning LangGraph! Let me know if you'd like to add any tasks or save more study notes about it.", 'extras': {'signature': 'El4KXAERTTIP9RGpX65N2TeWUdNOi7JOSYSXFl5yw86VoARyoGJSOWWkPUc+lnUoEnjdrOzlOtfspiWRHXc+qI4CLZSjnKbxL4avCP0cqLKZD6VovP5bWPGi+b+UAKb8'}}]


In [32]:
ask_agent(
    "Remember that I need to practice RAGAS tomorrow."
)

Agent:
[{'type': 'text', 'text': 'I\'ve added "Practice RAGAS tomorrow" to your task list!', 'extras': {'signature': 'El4KXAERTTIPTIENTZ59vGVo0Pu6irCXjege+6EQo9HO18rJmtOJXEZ8YZ8J0XbYyCJIHZd2VnsVBxvapX6+RzJThmcsbSiFSKpaj+P4kgnC2HN/Tgq+7Hh/r6BAUufn'}}]


In [34]:
ask_agent(
    "Add a task: Finish my Day 34 agent-memory homework."
)

Agent:
[{'type': 'text', 'text': 'I\'ve added "Finish my Day 34 agent-memory homework" to your task list.', 'extras': {'signature': 'El4KXAERTTIPi/u+q2e3GLjG23fcOtyZwdKyv8hx0j/ktZXKKZyJt2P1CQf+FrfaPLLuTiRsc5r8Ztc+ntNLlsIrA/PAOWRG9tkoXCKZWoPkywf3ES93A4EFt7dnH/jJ'}}]


In [35]:
ask_agent(
    "Add a task: Build a RAG application with Gemini."
)

Agent:
[{'type': 'text', 'text': 'I\'ve added "Build a RAG application with Gemini" to your tasks as well.', 'extras': {'signature': 'El4KXAERTTIPM4eXge/X5ZRkRbrO8NbmZK5MHuTqwOoy62Lm6+eQFgY05pCvJl11UOZ5SUU6GF1PjAW8mWi3bsiTRB7UiywPE6wf48eeedOl6fxF7dNeflF4Z+hcDBUy'}}]


In [36]:
ask_agent(
    "Show me my study tasks."
)

Agent:
[{'type': 'text', 'text': 'Here are your current study tasks:\n\n1. Practice RAGAS tomorrow\n2. Finish my Day 34 agent-memory homework\n3. Build a RAG application with Gemini', 'extras': {'signature': 'El4KXAERTTIPT8uvWIEfvpsNLb0/X5ZAZjt4aPZLHvqT8F3Skmmdmivdb5CgcjOYBu+IjBFYuMdYxALpN7NdNqWz+umZRtuTKTPVp+fh39BFqBCtniER1kCwRAkVdVuq'}}]


In [37]:
ask_agent(
    "Mark task 1 as completed."
)

Agent:
[{'type': 'text', 'text': 'I\'ve marked task 1 ("Practice RAGAS tomorrow") as completed.', 'extras': {'signature': 'El4KXAERTTIPsVExNGGlraBCDs4ftxlpX2FxHdNL1GjaZ2mlGuX5jnPk2WePlM/dV4N8IOqxNnO/sChpZr/I4yoryNt/cQ9Oyz6tykM8uWIhu/rgdGc9nVV85FR+C0Bi'}}]


In [38]:
ask_agent(
    "Show my tasks again."
)

Agent:
[{'type': 'text', 'text': 'Here is your updated task list:\n\n1. Practice RAGAS tomorrow [completed]\n2. Finish my Day 34 agent-memory homework [pending]\n3. Build a RAG application with Gemini [pending]', 'extras': {'signature': 'El4KXAERTTIPre9Wu4AS/IrX3riMurkG5KVOkALIyMLF0ntXxcnpZavqsDbnjpnqNOVYtCwZ2WGq170KjlJEvV7u+o9c8kdjxYux9gtfljNORLZpMi1myPP1TE/h9MZR'}}]
